In [1]:
# Import necessary libraries
# -------------------------------
import pandas as pd
import os
import time
from sqlalchemy import create_engine

In [4]:
# Create database connection
# This creates SQLite database file
# -------------------------------
engine = create_engine('sqlite:///inventory.db')
def ingest_db(df, table_name, engine):
    df.to_sql(name=table_name, con=engine, if_exists = 'append', index=False, chunksize=1000)


In [5]:
# Start timer to measure performance
# -------------------------------
start = time.time()

# Loop through all CSV files
# inside the data folder
# -------------------------------
for file in os.listdir('data'):
    if file.endswith('.csv'):
        file_path = os.path.join('data',file)
        print(f"\n Processing file : {file}")
        
        # -------------------------------
        # Read CSV in chunks
        # Only 100k rows loaded at a time
        # This prevents RAM overflow
        # -------------------------------
        chunk_size = 100000
        chunk_itr = pd.read_csv(
            file_path,
            chunksize = chunk_size,
            low_memory=True
        )

         # Process each chunk separately
        # -------------------------------
        for i, chunk in enumerate(chunk_itr):
            print(f"Inserting chunk {i+1} with shape {chunk.shape}")

            ingest_db(
                df=chunk,
                table_name = file[:-4],
                engine=engine
            )
            
# -------------------------------
# End timer
# -------------------------------
end = time.time()

total_time = (end - start) / 60

print("\nSaving Data Complete")
print(f"Time taken to save data: {total_time:.2f} minutes")


 Processing file : begin_inventory.csv
Inserting chunk 1 with shape (100000, 9)
Inserting chunk 2 with shape (100000, 9)
Inserting chunk 3 with shape (6529, 9)

 Processing file : end_inventory.csv
Inserting chunk 1 with shape (100000, 9)
Inserting chunk 2 with shape (100000, 9)
Inserting chunk 3 with shape (24489, 9)

 Processing file : purchases.csv
Inserting chunk 1 with shape (100000, 16)
Inserting chunk 2 with shape (100000, 16)
Inserting chunk 3 with shape (100000, 16)
Inserting chunk 4 with shape (100000, 16)
Inserting chunk 5 with shape (100000, 16)
Inserting chunk 6 with shape (100000, 16)
Inserting chunk 7 with shape (100000, 16)
Inserting chunk 8 with shape (100000, 16)
Inserting chunk 9 with shape (100000, 16)
Inserting chunk 10 with shape (100000, 16)
Inserting chunk 11 with shape (100000, 16)
Inserting chunk 12 with shape (100000, 16)
Inserting chunk 13 with shape (100000, 16)
Inserting chunk 14 with shape (100000, 16)
Inserting chunk 15 with shape (100000, 16)
Inserting

In [6]:
tables = [
    "begin_inventory",
    "end_inventory",
    "purchases",
    "purchase_prices",
    "sales",
    "vendor_invoice"
]
for table in tables:
    query = f"SELECT COUNT(*) as total_rows from {table}"
    df=pd.read_sql(query, engine)
    print(f"{table} :", df["total_rows"][0])

begin_inventory : 206529
end_inventory : 224489
purchases : 2372474
purchase_prices : 12261
sales : 12825363
vendor_invoice : 5543


In [7]:

for file in os.listdir("data"):

    if file.endswith(".csv"):

        table = file[:-4]
        path = os.path.join("data", file)

        # count CSV rows
        csv_rows = sum(1 for line in open(path)) - 1

        # count DB rows
        db_rows = pd.read_sql(
            f"SELECT COUNT(*) as c FROM {table}",
            engine
        )["c"][0]

        print(f"{table}")
        print(f"CSV rows : {csv_rows}")
        print(f"DB rows  : {db_rows}")
        print("-"*40)

begin_inventory
CSV rows : 206529
DB rows  : 206529
----------------------------------------
end_inventory
CSV rows : 224489
DB rows  : 224489
----------------------------------------
purchases
CSV rows : 2372474
DB rows  : 2372474
----------------------------------------
purchase_prices
CSV rows : 12261
DB rows  : 12261
----------------------------------------
sales
CSV rows : 12825363
DB rows  : 12825363
----------------------------------------
vendor_invoice
CSV rows : 5543
DB rows  : 5543
----------------------------------------
